In [2]:
"""
Normalizacion radiometrica para mosaico SAR (adaptacion practica del metodo RPRN,
Liu et al. 2022, IEEE GRSL, "Radiometric Principle-Based Radiometric Normalization
Method for SAR Images Mosaic").

Esta version es fiel al alcance del paper: SOLO corrige diferencias de MEDIA
(brillo) entre imagenes mediante un factor multiplicativo h_i por imagen. No se
intenta igualar la desviacion estandar/textura -- esa diferencia entre subswaths
adyacentes (IW1/IW2/IW3) responde en parte al angulo de incidencia distinto de
cada uno, que es fisicamente esperado y no debe eliminarse por la fuerza.

Adaptacion respecto al paper original:
- El paper trabaja con SLC sin geocodificar y usa RPCs + tie points para encontrar
  areas de cobertura consistente en espacio-objeto entre imagenes.
- Aqui las imagenes YA vienen con Terrain Correction (georreferenciadas), por lo que
  el area de solape se obtiene directamente de la interseccion geografica de cada
  GeoTIFF -- no hace falta paso de registro ni RPCs.
- El modelo de ajuste h_i * sigma_i = h_j * sigma_j se resuelve aqui en forma
  log-lineal en un solo paso de minimos cuadrados con regularizacion ridge
  (equivalente en el optimo al esquema iterativo del paper, pero sin iterar).

Pasos:
  3. Estadisticas locales (media/desv. estandar) por imagen en cada solape.
  4. Filtrado de "candidatos de ajuste": se descartan zonas de reflexion especular
     (media local baja -> agua, asfalto liso) y zonas de doble rebote
     (desv. estandar local alta -> zonas urbanas).
  5. Construccion y resolucion del sistema global de minimos cuadrados (ridge)
     para obtener un factor de correccion multiplicativo h_i por imagen.
  6. Aplicacion de h_i a cada imagen y exportacion, mas tabla de metricas
     antes/despues (Dmu, Ddelta, RMSE) igual al Table II del paper.
  Ademas imprime tiempos de ejecucion, estadisticas de pares, candidatos y
  ganancias, y las caracteristicas del equipo, y las guarda en rprn_report.json.

Entrada: las salidas de proceso1mosaico.ipynb (carpeta 2_PROCES), es decir, cada
<nombre>.dim con su carpeta <nombre>.data/Sigma0.img. Si la .img no trae la
georreferenciacion, se toma del .dim (WKT e IMAGE_TO_MODEL_TRANSFORM) mediante un
.vrt auxiliar, sin copiar los datos. Tambien acepta .tif si la carpeta no tiene .dim.
Salida: <nombre>_RN.tif (GeoTIFF float32, nodata=0, banda Sigma0; el nodata queda
marcado tambien para SNAP, asi el mosaico no promedia los bordes vacios) y rprn_report.json.

Requiere: numpy, scipy, rasterio
"""

from __future__ import annotations

import ctypes
import itertools
import json
import os
import platform
import re
import shutil
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from xml.sax.saxutils import escape

import numpy as np
import rasterio
from rasterio.errors import NotGeoreferencedWarning
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from scipy.ndimage import uniform_filter


# --------------------------------------------------------------------------
# Configuracion
# --------------------------------------------------------------------------

@dataclass
class Config:
    window_size: int = 9          # ventana para estadisticas locales (impar)
    low_mean_pct: float = 15.0    # percentil bajo de media -> descartar (especular)
    high_std_pct: float = 85.0    # percentil alto de std -> descartar (doble rebote)
    min_candidates: int = 500     # minimo de pixeles candidatos validos por par
    ridge_lambda: float = 1e-3    # regularizacion del sistema de minimos cuadrados
    nodata_value: float | None = 0.0  # valor a tratar como nodata (ajustar si aplica)
    band_name: str = "Sigma0"     # banda que deja proceso1mosaico en <nombre>.data/
    excluir_nodata_al_reproyectar: bool = True  # el nodata no se mezcla en la interpolacion bilineal
    chunk_rows: int = 1024        # filas por bloque al escribir las imagenes corregidas


# --------------------------------------------------------------------------
# Entrada: salidas de proceso1mosaico (.dim -> .data/Sigma0.img)
# --------------------------------------------------------------------------

_TIPOS_GDAL = {"uint8": "Byte", "uint16": "UInt16", "int16": "Int16", "uint32": "UInt32",
               "int32": "Int32", "float32": "Float32", "float64": "Float64"}


def _crear_vrt(dim_path: Path, img_path: Path, carpeta: Path) -> str:
    """Crea un .vrt que apunta a la .img y toma la georreferenciacion del .dim
    (WKT e IMAGE_TO_MODEL_TRANSFORM). No copia los datos."""
    txt = dim_path.read_text(encoding="latin-1", errors="ignore")
    wkt = re.search(r"<WKT>(.*?)</WKT>", txt, re.S)
    tr = re.search(r"<IMAGE_TO_MODEL_TRANSFORM>(.*?)</IMAGE_TO_MODEL_TRANSFORM>", txt)
    if not (wkt and tr):
        raise ValueError(f"{dim_path.name}: no tiene georreferenciacion (WKT / IMAGE_TO_MODEL_TRANSFORM).")
    # Orden de SNAP (AffineTransform de Java): m00, m10, m01, m11, m02, m12
    m00, m10, m01, m11, m02, m12 = [float(v) for v in tr.group(1).split(",")]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", NotGeoreferencedWarning)
        with rasterio.open(img_path) as s:
            ancho, alto, tipo = s.width, s.height, _TIPOS_GDAL[s.dtypes[0]]
    carpeta.mkdir(parents=True, exist_ok=True)
    vrt = carpeta / f"{dim_path.stem}.vrt"
    vrt.write_text(
        f'<VRTDataset rasterXSize="{ancho}" rasterYSize="{alto}">\n'
        f'  <SRS>{escape(" ".join(wkt.group(1).split()))}</SRS>\n'
        f'  <GeoTransform>{m02!r}, {m00!r}, {m01!r}, {m12!r}, {m10!r}, {m11!r}</GeoTransform>\n'
        f'  <VRTRasterBand dataType="{tipo}" band="1">\n'
        f'    <SimpleSource>\n'
        f'      <SourceFilename relativeToVRT="0">{escape(str(img_path.resolve()))}</SourceFilename>\n'
        f'      <SourceBand>1</SourceBand>\n'
        f'    </SimpleSource>\n'
        f'  </VRTRasterBand>\n'
        f'</VRTDataset>\n', encoding="utf-8")
    return str(vrt)


def preparar_entradas(entradas: list[str], output_dir: Path, cfg: Config):
    """Para cada .dim devuelve la ruta legible de su banda Sigma0 (o un .vrt si la
    .img no trae georreferenciacion). Los .tif se usan tal cual.
    Devuelve (nombres, rutas, info por imagen)."""
    nombres, rutas, info = [], [], []
    for entrada in entradas:
        p = Path(entrada)
        origen = "tif"
        ruta = str(p)
        if p.suffix.lower() == ".dim":
            img = p.with_suffix(".data") / f"{cfg.band_name}.img"
            if not img.exists():
                disponibles = [q.stem for q in p.with_suffix(".data").glob("*.img")]
                raise FileNotFoundError(
                    f"No existe {img}. Bandas disponibles: {disponibles}. "
                    f"Procese las imagenes con proceso1mosaico.ipynb, que renombra la banda a {cfg.band_name}.")
            ruta, origen = str(img), "dim"
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", NotGeoreferencedWarning)
                with rasterio.open(ruta) as s:
                    sin_georef = s.crs is None or s.transform.is_identity
            if sin_georef:
                ruta, origen = _crear_vrt(p, img, output_dir / "_vrt_entrada"), "dim (georref. del .dim)"
        with rasterio.open(ruta) as s:
            if s.crs is None:
                raise ValueError(f"{entrada}: sin georreferenciacion (CRS).")
            info.append({"nombre": p.stem, "origen": origen, "columnas": s.width, "filas": s.height,
                         "tipo": s.dtypes[0], "crs": s.crs.to_string(),
                         "pixel_x": abs(s.transform.a), "pixel_y": abs(s.transform.e)})
        nombres.append(p.stem)
        rutas.append(ruta)
    return nombres, rutas, info


# --------------------------------------------------------------------------
# Caracteristicas del equipo (si algun dato no se puede leer, se muestra "-")
# --------------------------------------------------------------------------

def _ram_gb():
    """RAM total y disponible en GB."""
    try:
        import psutil
        vm = psutil.virtual_memory()
        return vm.total / 1024 ** 3, vm.available / 1024 ** 3
    except Exception:
        pass
    try:
        if sys.platform.startswith("win"):
            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                            ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                            ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                            ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                            ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
            m = MEMORYSTATUSEX()
            m.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            ctypes.WinDLL("kernel32").GlobalMemoryStatusEx(ctypes.byref(m))
            return m.ullTotalPhys / 1024 ** 3, m.ullAvailPhys / 1024 ** 3
        with open("/proc/meminfo") as f:
            datos = {l.split(":")[0]: int(l.split()[1]) for l in f if ":" in l}
        return datos["MemTotal"] / 1024 ** 2, datos.get("MemAvailable", 0) / 1024 ** 2
    except Exception:
        return None, None


def _nombre_cpu():
    """Nombre comercial del procesador."""
    try:
        if sys.platform.startswith("win"):
            import winreg
            clave = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(clave, "ProcessorNameString")[0].strip()
        with open("/proc/cpuinfo") as f:
            for l in f:
                if l.lower().startswith("model name"):
                    return l.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "-"


def _memoria_pico_gb():
    """Memoria maxima usada por este proceso de Python, en GB."""
    try:
        import psutil
        mi = psutil.Process().memory_info()
        if hasattr(mi, "peak_wset"):
            return mi.peak_wset / 1024 ** 3
    except Exception:
        pass
    try:
        if sys.platform.startswith("win"):
            class PMC(ctypes.Structure):
                _fields_ = [("cb", ctypes.c_ulong), ("PageFaultCount", ctypes.c_ulong),
                            ("PeakWorkingSetSize", ctypes.c_size_t), ("WorkingSetSize", ctypes.c_size_t),
                            ("QuotaPeakPagedPoolUsage", ctypes.c_size_t), ("QuotaPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t), ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                            ("PagefileUsage", ctypes.c_size_t), ("PeakPagefileUsage", ctypes.c_size_t)]
            k32, psapi = ctypes.WinDLL("kernel32"), ctypes.WinDLL("psapi")
            k32.GetCurrentProcess.restype = ctypes.c_void_p
            psapi.GetProcessMemoryInfo.argtypes = [ctypes.c_void_p, ctypes.POINTER(PMC), ctypes.c_ulong]
            c = PMC()
            c.cb = ctypes.sizeof(PMC)
            if psapi.GetProcessMemoryInfo(k32.GetCurrentProcess(), ctypes.byref(c), c.cb):
                return c.PeakWorkingSetSize / 1024 ** 3
            return None
        import resource
        pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return pico / 1024 ** 3 if sys.platform == "darwin" else pico / 1024 ** 2
    except Exception:
        return None


def info_equipo(carpeta: Path) -> dict:
    """Caracteristicas del equipo y versiones de las librerias."""
    total, disponible = _ram_gb()
    try:
        libre = shutil.disk_usage(carpeta).free / 1024 ** 3
    except Exception:
        libre = None
    try:
        import scipy
        v_scipy = scipy.__version__
    except Exception:
        v_scipy = "-"
    return {
        "equipo": platform.node(),
        "sistema": f"{platform.system()} {platform.release()} ({platform.machine()})",
        "procesador": _nombre_cpu(),
        "nucleos_logicos": os.cpu_count(),
        "ram_total_gb": round(total, 1) if total else None,
        "ram_disponible_inicio_gb": round(disponible, 1) if disponible else None,
        "disco_libre_salida_gb": round(libre, 1) if libre else None,
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": v_scipy,
        "rasterio": rasterio.__version__,
        "gdal": getattr(rasterio, "__gdal_version__", "-"),
    }


def _imprimir_equipo(eq: dict):
    print(" Equipo:", eq["equipo"], "|", eq["sistema"])
    print(" Procesador:", eq["procesador"], "|", eq["nucleos_logicos"], "nucleos logicos")
    print(f" RAM: total {eq['ram_total_gb'] or '-'} GB | disponible al inicio {eq['ram_disponible_inicio_gb'] or '-'} GB")
    print(f" Disco libre en la carpeta de salida: {eq['disco_libre_salida_gb'] or '-'} GB")
    print(f" Python {eq['python']} | numpy {eq['numpy']} | scipy {eq['scipy']} | "
          f"rasterio {eq['rasterio']} | GDAL {eq['gdal']}")


# --------------------------------------------------------------------------
# Paso 3: lectura + estadisticas locales
# --------------------------------------------------------------------------

def local_mean_std(arr: np.ndarray, window: int, valid_mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Media y desviacion estandar local via filtro de ventana deslizante,
    excluyendo pixeles invalidos (nodata) del calculo."""
    arr_filled = np.where(valid_mask, arr, 0.0)
    valid_f = valid_mask.astype(np.float64)

    count = uniform_filter(valid_f, size=window, mode="nearest")
    count = np.clip(count, 1e-6, None)

    mean = uniform_filter(arr_filled, size=window, mode="nearest") / count
    mean_sq = uniform_filter(arr_filled ** 2, size=window, mode="nearest") / count
    var = np.clip(mean_sq - mean ** 2, 0, None)
    std = np.sqrt(var)
    return mean, std


def overlap_arrays_on_grid_i(path_i: str, path_j: str, cfg: Config):
    """Reproyecta la imagen j sobre la grilla de la imagen i, recorta al bbox
    de solape y devuelve ambos arrays alineados pixel a pixel."""
    with rasterio.open(path_i) as src_i, rasterio.open(path_j) as src_j:
        bounds_i = src_i.bounds
        bounds_j = src_j.bounds

        left = max(bounds_i.left, bounds_j.left)
        right = min(bounds_i.right, bounds_j.right)
        bottom = max(bounds_i.bottom, bounds_j.bottom)
        top = min(bounds_i.top, bounds_j.top)

        if left >= right or bottom >= top:
            return None  # sin solape

        window_i = rasterio.windows.from_bounds(left, bottom, right, top, transform=src_i.transform)
        window_i = window_i.round_offsets().round_lengths()
        if window_i.width < 2 or window_i.height < 2:
            return None

        arr_i = src_i.read(1, window=window_i).astype(np.float64)
        transform_overlap = src_i.window_transform(window_i)

        arr_j = np.full(arr_i.shape, np.nan, dtype=np.float64)
        nodata_extra = {}
        if cfg.excluir_nodata_al_reproyectar and cfg.nodata_value is not None:
            nodata_extra = {"src_nodata": cfg.nodata_value, "dst_nodata": np.nan}
        reproject(
            source=rasterio.band(src_j, 1),
            destination=arr_j,
            src_transform=src_j.transform,
            src_crs=src_j.crs,
            dst_transform=transform_overlap,
            dst_crs=src_i.crs,
            resampling=Resampling.bilinear,
            **nodata_extra,
        )

        return arr_i, arr_j


# --------------------------------------------------------------------------
# Paso 4: seleccion de candidatos (excluir especular y doble rebote)
# --------------------------------------------------------------------------

def select_candidates(arr_i, arr_j, cfg: Config, stats: dict | None = None):
    """Filtro de 'candidatos de ajuste' del paper: descarta reflexion especular
    (media local baja) y doble rebote (desv. estandar local alta) en ambas
    imagenes, quedandose solo con terreno rugoso estable.
    Si se pasa 'stats', anota cuantos pixeles descarta cada criterio."""
    nodata = cfg.nodata_value
    valid = np.isfinite(arr_i) & np.isfinite(arr_j)
    if nodata is not None:
        valid &= (arr_i != nodata) & (arr_j != nodata)
    valid &= (arr_i > 0) & (arr_j > 0)  # sigma0 lineal debe ser positivo
    if stats is not None:
        stats["n_validos"] = int(valid.sum())

    if valid.sum() < cfg.min_candidates:
        return None

    mean_i, std_i = local_mean_std(arr_i, cfg.window_size, valid)
    mean_j, std_j = local_mean_std(arr_j, cfg.window_size, valid)

    low_i = np.percentile(mean_i[valid], cfg.low_mean_pct)
    low_j = np.percentile(mean_j[valid], cfg.low_mean_pct)
    high_i = np.percentile(std_i[valid], cfg.high_std_pct)
    high_j = np.percentile(std_j[valid], cfg.high_std_pct)

    stable = (
        valid
        & (mean_i > low_i) & (mean_j > low_j)   # descarta especular
        & (std_i < high_i) & (std_j < high_j)   # descarta doble rebote
    )

    if stats is not None:
        sin_especular = valid & (mean_i > low_i) & (mean_j > low_j)
        stats["n_especular"] = int(valid.sum() - sin_especular.sum())
        stats["n_doble_rebote"] = int(sin_especular.sum() - stable.sum())
        stats["n_candidatos"] = int(stable.sum())

    if stable.sum() < cfg.min_candidates:
        return None

    return arr_i[stable], arr_j[stable]


# --------------------------------------------------------------------------
# Paso 5: sistema global de minimos cuadrados (ganancias h_i)
# --------------------------------------------------------------------------

@dataclass
class PairStats:
    idx_i: int
    idx_j: int
    log_ratio: float   # log( mediana(sigma_j) / mediana(sigma_i) ) sobre candidatos
    n_candidates: int
    mean_i: float
    mean_j: float
    std_i: float
    std_j: float


def solve_gains(pairs: list[PairStats], n_images: int, cfg: Config) -> np.ndarray:
    """Resuelve log(h_i) - log(h_j) = log_ratio_ij para todas las parejas a la vez
    (ajuste conjunto, no en cadena), con regularizacion ridge para fijar el
    nivel de referencia (el sistema es indeterminado ante un desplazamiento
    global constante de todos los log(h_i))."""
    rows = []
    rhs = []
    for p in pairs:
        row = np.zeros(n_images)
        row[p.idx_i] = 1.0
        row[p.idx_j] = -1.0
        rows.append(row)
        rhs.append(p.log_ratio)

    A = np.array(rows)
    L = np.array(rhs)

    AtA = A.T @ A + cfg.ridge_lambda * np.eye(n_images)
    AtL = A.T @ L
    log_h = np.linalg.solve(AtA, AtL)
    log_h -= log_h.mean()  # ancla: referencia = nivel medio del conjunto
    return np.exp(log_h)


def _registrar_par(stats, i, j, estado, info, t0):
    """Guarda el resultado de un par para el resumen (no afecta el calculo)."""
    if stats is None:
        return
    registro = {"i": i, "j": j, "estado": estado, "tiempo_s": time.perf_counter() - t0}
    registro.update(info or {})
    stats.setdefault("pares", []).append(registro)


def build_pairs(image_paths: list[str], cfg: Config, stats: dict | None = None) -> list[PairStats]:
    pairs = []
    for (i, path_i), (j, path_j) in itertools.combinations(enumerate(image_paths), 2):
        t0 = time.perf_counter()
        info = {} if stats is not None else None
        res = overlap_arrays_on_grid_i(path_i, path_j, cfg)
        if res is None:
            _registrar_par(stats, i, j, "sin traslape", info, t0)
            continue
        arr_i, arr_j = res
        if info is not None:
            info["n_traslape"] = int(arr_i.size)
        cand = select_candidates(arr_i, arr_j, cfg, info)
        if cand is None:
            estado = "pocos candidatos" if info and "n_candidatos" in info else "traslape sin datos"
            _registrar_par(stats, i, j, estado, info, t0)
            continue
        vals_i, vals_j = cand
        ratio = np.median(vals_j) / np.median(vals_i)
        if not np.isfinite(ratio) or ratio <= 0:
            _registrar_par(stats, i, j, "razon invalida", info, t0)
            continue
        pairs.append(PairStats(
            idx_i=i, idx_j=j,
            log_ratio=float(np.log(ratio)),
            n_candidates=int(vals_i.size),
            mean_i=float(vals_i.mean()), mean_j=float(vals_j.mean()),
            std_i=float(vals_i.std()), std_j=float(vals_j.std()),
        ))
        _registrar_par(stats, i, j, "utilizable", info, t0)
    return pairs


# --------------------------------------------------------------------------
# Paso 6: aplicar ganancias, exportar, y metricas antes/despues
# --------------------------------------------------------------------------

def apply_gain_and_save(path_in: str, gain: float, path_out: str, cfg: Config | None = None):
    """Escribe la imagen corregida (valor * h_i) como GeoTIFF float32, por bloques de
    filas para no cargar la imagen completa en memoria."""
    cfg = cfg or Config()
    with rasterio.open(path_in) as src:
        profile = dict(driver="GTiff", height=src.height, width=src.width, count=1, dtype="float32",
                       crs=src.crs, transform=src.transform, nodata=0.0, compress="deflate",
                       tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
        with rasterio.open(path_out, "w", **profile) as dst:
            # SNAP (hasta la version 12) no lee la etiqueta nodata estandar del GeoTIFF, sino
            # estos metadatos. Sin ellos toma los 0 de fuera de la huella como datos y en el
            # mosaico los promedia con las imagenes vecinas (rectangulos oscuros en los solapes).
            dst.update_tags(1, BAND=cfg.band_name, NODATA="0")
            for r0 in range(0, src.height, cfg.chunk_rows):
                win = Window(0, r0, src.width, min(cfg.chunk_rows, src.height - r0))
                arr = src.read(1, window=win).astype(np.float64)
                valido = np.isfinite(arr)
                if cfg.nodata_value is not None:
                    valido &= arr != cfg.nodata_value
                corrected = np.where(valido, arr * gain, 0.0)  # el nodata de la entrada queda en 0
                dst.write(corrected.astype(np.float32), 1, window=win)


def compute_metrics(pairs: list[PairStats], gains: np.ndarray, label: str):
    """Replica los indicadores del Table II del paper (Dmu, Ddelta, RMSE),
    calculados antes (gains=1) y despues de aplicar las ganancias."""
    d_mu, d_delta, sq_errors = [], [], []
    for p in pairs:
        m_i = p.mean_i * gains[p.idx_i]
        m_j = p.mean_j * gains[p.idx_j]
        s_i = p.std_i * gains[p.idx_i]
        s_j = p.std_j * gains[p.idx_j]
        d_mu.append(abs(m_i - m_j))
        d_delta.append(abs(s_i - s_j))
        sq_errors.append((m_i - m_j) ** 2)
    d_mu = float(np.mean(d_mu)) if d_mu else float("nan")
    d_delta = float(np.mean(d_delta)) if d_delta else float("nan")
    rmse = float(np.sqrt(np.mean(sq_errors))) if sq_errors else float("nan")
    return {"label": label, "D_mu": d_mu, "D_delta": d_delta, "RMSE": rmse}


# --------------------------------------------------------------------------
# Estadisticas de ejecucion (solo informan; no cambian el calculo)
# --------------------------------------------------------------------------

def _fmt_tiempo(seg: float) -> str:
    """Formatea segundos como h:mm:ss."""
    seg = int(round(seg))
    return f"{seg // 3600}:{(seg % 3600) // 60:02d}:{seg % 60:02d}"


def _dif_relativa_pct(m_i: float, m_j: float) -> float:
    """Diferencia de brillo medio entre dos imagenes, en % de su promedio
    (no depende de la escala de los datos)."""
    prom = (m_i + m_j) / 2
    return float(abs(m_i - m_j) / prom * 100) if prom > 0 else float("nan")


def _cambio_pct(antes: float, despues: float) -> float:
    return float((despues - antes) / antes * 100) if antes else float("nan")


# --------------------------------------------------------------------------
# Orquestacion
# --------------------------------------------------------------------------

def run(image_paths: list[str], output_dir: str, cfg: Config | None = None):
    t_inicio = time.perf_counter()
    tiempos = {}
    cfg = cfg or Config()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fecha_inicio = time.strftime("%Y-%m-%d %H:%M:%S")
    print("Inicio:", fecha_inicio)
    equipo = info_equipo(output_dir)
    _imprimir_equipo(equipo)
    print(f"Parametros: ventana {cfg.window_size}x{cfg.window_size} | percentil media {cfg.low_mean_pct} | "
          f"percentil desv. {cfg.high_std_pct} | min. candidatos {cfg.min_candidates} | lambda {cfg.ridge_lambda}")

    # Entradas: .dim de proceso1mosaico (banda Sigma0) o .tif
    t0 = time.perf_counter()
    nombres, image_paths, entradas = preparar_entradas(image_paths, output_dir, cfg)
    tiempos["0_lectura_entradas"] = time.perf_counter() - t0
    origenes = sorted({e["origen"] for e in entradas})
    tipos = sorted({e["tipo"] for e in entradas})
    crs_distintos = sorted({e["crs"] for e in entradas})
    px = [e["pixel_x"] for e in entradas]
    mpx = sum(e["columnas"] * e["filas"] for e in entradas) / 1e6
    print(f"Entradas: {len(entradas)} imagenes | origen: {', '.join(origenes)} | tipo de dato: {', '.join(tipos)}")
    print(f"  Tamano de pixel: {min(px):.6f} a {max(px):.6f} grados (~{min(px) * 111320:.0f} a {max(px) * 111320:.0f} m)"
          f" | {mpx:.1f} millones de pixeles en total")
    if len(crs_distintos) > 1:
        print("  AVISO: las imagenes no tienen el mismo sistema de referencia:", crs_distintos)

    print(f"[1/4] Construyendo pares de solape y candidatos para {len(image_paths)} imagenes...")
    t0 = time.perf_counter()
    stats_pares = {}
    pairs = build_pairs(image_paths, cfg, stats_pares)
    tiempos["1_pares_y_candidatos"] = time.perf_counter() - t0
    if not pairs:
        raise RuntimeError(
            "No se encontraron pares con solape valido y suficientes candidatos. "
            "Revisa CRS/bounds de los GeoTIFF o baja 'min_candidates'."
        )
    print(f"      {len(pairs)} pares utilizables de {len(image_paths) * (len(image_paths) - 1) // 2} posibles.")

    registros = stats_pares.get("pares", [])
    estados = ["utilizable", "sin traslape", "traslape sin datos", "pocos candidatos", "razon invalida"]
    conteo = {e: sum(1 for r in registros if r["estado"] == e) for e in estados}
    print("      Clasificacion de los pares:")
    for e in estados:
        print(f"        {e:20s} {conteo[e]:4d}")
    usados = [r for r in registros if r["estado"] == "utilizable"]
    n_val = sum(r.get("n_validos", 0) for r in usados)
    n_esp = sum(r.get("n_especular", 0) for r in usados)
    n_dob = sum(r.get("n_doble_rebote", 0) for r in usados)
    n_cand = sum(r.get("n_candidatos", 0) for r in usados)
    if n_val:
        print(f"      Pixeles validos en los traslapes utilizables: {n_val:,}")
        print(f"        descartados por reflexion especular (agua): {n_esp:,} ({n_esp / n_val * 100:.1f} %)")
        print(f"        descartados por doble rebote (ciudad):      {n_dob:,} ({n_dob / n_val * 100:.1f} %)")
        print(f"        candidatos de ajuste:                       {n_cand:,} ({n_cand / n_val * 100:.1f} %)")
    if registros:
        print(f"      Tiempo: {_fmt_tiempo(tiempos['1_pares_y_candidatos'])} "
              f"(promedio {tiempos['1_pares_y_candidatos'] / len(registros):.1f} s por par evaluado)")

    connected = {p.idx_i for p in pairs} | {p.idx_j for p in pairs}
    isolated = [nombres[i] for i in range(len(image_paths)) if i not in connected]
    if isolated:
        print("      AVISO: estas imagenes no tuvieron NINGUN par de solape valido")
        print("      y quedaran sin corregir de forma confiable:")
        for p in isolated:
            print(f"        - {p}")

    print("[2/4] Resolviendo sistema global de ganancias (minimos cuadrados ridge)...")
    t0 = time.perf_counter()
    gains = solve_gains(pairs, len(image_paths), cfg)
    tiempos["2_ajuste_global"] = time.perf_counter() - t0
    for nombre, g in zip(nombres, gains):
        print(f"      h[{nombre}] = {g:.4f}")
    pares_por_imagen = [sum(1 for p in pairs if i in (p.idx_i, p.idx_j)) for i in range(len(image_paths))]
    print(f"      Ganancias: minima {gains.min():.4f} | maxima {gains.max():.4f} | "
          f"media {gains.mean():.4f} | desv. {gains.std():.4f}")
    print("      Por imagen: [#] ganancia, cambio de brillo, pares usados, nombre")
    for i, (nombre, g) in enumerate(zip(nombres, gains)):
        print(f"        [{i:2d}] h={g:.4f} {(g - 1) * 100:+7.1f} % {pares_por_imagen[i]:3d} pares  {nombre}")
    print(f"      Tiempo: {tiempos['2_ajuste_global']:.2f} s")

    print("[3/4] Calculando metricas antes/despues...")
    t0 = time.perf_counter()
    gains_before = np.ones_like(gains)
    metrics_before = compute_metrics(pairs, gains_before, "antes (original)")
    metrics_after = compute_metrics(pairs, gains, "despues (RPRN)")
    for m in (metrics_before, metrics_after):
        print(f"      {m['label']:22s} D_mu={m['D_mu']:.4f}  D_delta={m['D_delta']:.4f}  RMSE={m['RMSE']:.4f}")
    cambio = {k: _cambio_pct(metrics_before[k], metrics_after[k]) for k in ("D_mu", "D_delta", "RMSE")}
    print("      Cambio de las metricas (despues frente a antes):")
    for k in ("D_mu", "D_delta", "RMSE"):
        print(f"        {k:8s} {metrics_before[k]:.4f} -> {metrics_after[k]:.4f}  ({cambio[k]:+.1f} %)")
    dif_antes = [_dif_relativa_pct(p.mean_i, p.mean_j) for p in pairs]
    dif_despues = [_dif_relativa_pct(p.mean_i * gains[p.idx_i], p.mean_j * gains[p.idx_j]) for p in pairs]
    print("      Diferencia de brillo medio entre imagenes vecinas (% del promedio del par):")
    print(f"        promedio: antes {np.nanmean(dif_antes):.1f} % -> despues {np.nanmean(dif_despues):.1f} %")
    print(f"        maxima:   antes {np.nanmax(dif_antes):.1f} % -> despues {np.nanmax(dif_despues):.1f} %")
    print("      Detalle por par: [i]-[j], candidatos, % de los pixeles validos, dif. antes -> despues")
    reg_por_par = {(r["i"], r["j"]): r for r in usados}
    for p, da, dd in zip(pairs, dif_antes, dif_despues):
        r = reg_por_par.get((p.idx_i, p.idx_j), {})
        pct = r.get("n_candidatos", 0) / r["n_validos"] * 100 if r.get("n_validos") else float("nan")
        print(f"        [{p.idx_i:2d}]-[{p.idx_j:2d}] {p.n_candidates:12,d} {pct:6.1f} %   {da:6.1f} % -> {dd:6.1f} %")
    tiempos["3_metricas"] = time.perf_counter() - t0

    print("[4/4] Aplicando ganancias y exportando imagenes corregidas...")
    t0 = time.perf_counter()
    out_paths = []
    exportadas = []
    for nombre, path, g in zip(nombres, image_paths, gains):
        out_path = output_dir / f"{nombre}_RN.tif"
        t_img = time.perf_counter()
        apply_gain_and_save(path, g, str(out_path), cfg)
        out_paths.append(str(out_path))
        print(f"      -> {out_path}")
        try:
            with rasterio.open(out_path) as s:
                cols, rows = s.width, s.height
            mb = out_path.stat().st_size / 1024 ** 2
            seg = time.perf_counter() - t_img
            exportadas.append({"imagen": out_path.name, "columnas": cols, "filas": rows,
                               "mb": round(mb, 1), "tiempo_s": round(seg, 1)})
            print(f"         {cols} x {rows} pixeles | {mb:.1f} MB | {seg:.1f} s")
        except Exception:
            pass
    tiempos["4_exportacion"] = time.perf_counter() - t0
    tiempos["total"] = time.perf_counter() - t_inicio
    pico = _memoria_pico_gb()
    _, disponible_fin = _ram_gb()

    report = {
        "gains": {n: float(g) for n, g in zip(nombres, gains)},
        "n_pairs_used": len(pairs),
        "isolated_images": isolated,
        "metrics_before": metrics_before,
        "metrics_after": metrics_after,
        "output_images": out_paths,
        "parametros": dict(vars(cfg)),
        "tiempos_s": {k: round(float(v), 2) for k, v in tiempos.items()},
        "clasificacion_pares": conteo,
        "pixeles_traslapes_utilizables": {"validos": n_val, "especular": n_esp,
                                          "doble_rebote": n_dob, "candidatos": n_cand},
        "cambio_metricas_pct": cambio,
        "diferencia_brillo_pct": {"antes_promedio": float(np.nanmean(dif_antes)),
                                  "despues_promedio": float(np.nanmean(dif_despues)),
                                  "antes_maxima": float(np.nanmax(dif_antes)),
                                  "despues_maxima": float(np.nanmax(dif_despues))},
        "ganancias_resumen": {"minima": float(gains.min()), "maxima": float(gains.max()),
                              "media": float(gains.mean()), "desv": float(gains.std())},
        "pares_por_imagen": {nom: int(n) for nom, n in zip(nombres, pares_por_imagen)},
        "detalle_pares": registros,
        "imagenes_exportadas": exportadas,
        "entradas": entradas,
        "inicio": fecha_inicio,
        "fin": time.strftime("%Y-%m-%d %H:%M:%S"),
        "equipo": equipo,
        "memoria_pico_gb": round(pico, 2) if pico else None,
        "ram_disponible_fin_gb": round(disponible_fin, 1) if disponible_fin else None,
    }
    report_path = output_dir / "rprn_report.json"
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"\nReporte guardado en: {report_path}")

    n_img = len(image_paths)
    print("\n" + "=" * 70)
    print(" RESUMEN DE EJECUCION - PROCESO 2 (normalizacion radiometrica)")
    print("=" * 70)
    print(" Fin:", time.strftime("%Y-%m-%d %H:%M:%S"))
    print(f" Imagenes: {n_img} | pares posibles: {n_img * (n_img - 1) // 2} | "
          f"utilizables: {len(pairs)} | imagenes sin par: {len(isolated)}")
    if n_val:
        print(f" Candidatos de ajuste: {n_cand:,} de {n_val:,} pixeles validos ({n_cand / n_val * 100:.1f} %)")
        print(f"   agua (especular) {n_esp / n_val * 100:.1f} % | ciudad (doble rebote) {n_dob / n_val * 100:.1f} %")
    print(f" Ganancias h: de {gains.min():.4f} a {gains.max():.4f}")
    for k in ("D_mu", "D_delta", "RMSE"):
        print(f" {k:8s} {metrics_before[k]:.4f} -> {metrics_after[k]:.4f} ({cambio[k]:+.1f} %)")
    print(f" Diferencia de brillo entre vecinas: {np.nanmean(dif_antes):.1f} % -> {np.nanmean(dif_despues):.1f} %")
    print(f" Tiempos: lectura {_fmt_tiempo(tiempos['0_lectura_entradas'])} | "
          f"pares y candidatos {_fmt_tiempo(tiempos['1_pares_y_candidatos'])} | "
          f"ajuste {tiempos['2_ajuste_global']:.2f} s | metricas {tiempos['3_metricas']:.2f} s | "
          f"exportacion {_fmt_tiempo(tiempos['4_exportacion'])}")
    print(f" Tiempo total: {_fmt_tiempo(tiempos['total'])} ({tiempos['total'] / 60:.1f} min)")
    print("\n Equipo donde se proceso:")
    _imprimir_equipo(equipo)
    print(f" Memoria maxima usada por el proceso: {pico:.1f} GB" if pico else " Memoria maxima usada: -")
    print(f" RAM disponible al final: {disponible_fin:.1f} GB" if disponible_fin else " RAM disponible al final: -")
    print("=" * 70)
    return report


if __name__ == "__main__":
    # CARPETA DE SALIDA DE proceso1mosaico.ipynb (su output_dir): .dim + .data/Sigma0.img
    INPUT_DIR = r"C:\argenis\RADAR\MOSAICO\2_PROCES"
    # CARPETA DE SALIDA DE LAS IMAGENES NORMALIZADAS (_RN.tif) Y DEL REPORTE
    OUTPUT_DIR = r"C:\argenis\RADAR\MOSAICO\3_CALIB"
    ARCHIVOS = sorted(Path(INPUT_DIR).glob("*.dim")) or sorted(Path(INPUT_DIR).glob("*.tif"))
    if not ARCHIVOS:
        raise FileNotFoundError(f"No hay archivos .dim ni .tif en {INPUT_DIR}")
    run([str(p) for p in ARCHIVOS], OUTPUT_DIR)

[1/4] Construyendo pares de solape y candidatos para 12 imagenes...
      19 pares utilizables de 66 posibles.
[2/4] Resolviendo sistema global de ganancias (minimos cuadrados ridge)...
      h[S1A_IW1_SLC__1SDV_20180823T101751_023372_028AF1_7881.tif] = 0.9302
      h[S1A_IW1_SLC__1SDV_20180823T101816_023372_028AF1_3153.tif] = 0.8714
      h[S1A_IW1_SLC__1SDV_20180823T101841_023372_028AF1_77B0.tif] = 1.0212
      h[S1A_IW1_SLC__1SDV_20180830T101056_023474_028E3B_0188.tif] = 1.8828
      h[S1A_IW2_SLC__1SDV_20180823T101751_023372_028AF1_7881.tif] = 0.6675
      h[S1A_IW2_SLC__1SDV_20180823T101816_023372_028AF1_3153.tif] = 0.7610
      h[S1A_IW2_SLC__1SDV_20180823T101841_023372_028AF1_77B0.tif] = 1.1035
      h[S1A_IW2_SLC__1SDV_20180830T101056_023474_028E3B_0188.tif] = 1.2633
      h[S1A_IW3_SLC__1SDV_20180823T101751_023372_028AF1_7881.tif] = 0.7365
      h[S1A_IW3_SLC__1SDV_20180823T101816_023372_028AF1_3153.tif] = 0.9085
      h[S1A_IW3_SLC__1SDV_20180823T101841_023372_028AF1_77B0.tif